# Karpathy-split evaluation — LEAKAGE-CHECKED version

This notebook is a **corrected replacement** for `evaluate_karpathy.ipynb`.

The original notebook resolved Karpathy-test images by looking them up in
`captions_train2017.json` / `captions_val2017.json` — the **same annotation
files the model was trained on**. Because COCO2017 simply redrew the
train/val boundary over the same COCO2014 image pool, a large share of the
images Karpathy calls "test" turned out to already be sitting inside
`train2017`. That means the model had almost certainly already seen those
images (and their captions) during training — so the earlier scores
(BLEU‑4 0.4165, CIDEr 1.276) mostly measured **memorization, not
generalization**.

This notebook fixes that by:

1. **Detecting contamination** — explicitly checking which Karpathy-test
   images overlap with the model's actual training annotation
   (`captions_train2017.json`).
2. **Excluding every contaminated image** from evaluation. The model is
   only ever scored on images it never saw during training.
3. Fetching the held-out images from a clean source: `val2017` first
   (never used for training here), then falling back to the **official
   COCO2014** CDN (using the `filepath`/`filename` fields that ship inside
   Karpathy's own `dataset_coco.json`) for any image that isn't present in
   either 2017 annotation file at all.
4. Scoring with the same official `pycocoevalcap` toolkit as before, so the
   *metric computation itself* is unchanged — only the image set going into
   it is now leak-free.
5. Reporting **both** numbers side by side — the old contaminated score and
   the new clean score — so the gap is visible and honest.

## Requirements (one-time)

```bash
pip install pycocoevalcap
```

You also need a **Java runtime** on your PATH (`pycocoevalcap`'s tokenizer
and METEOR scorer shell out to bundled Stanford NLP `.jar` files):

```bash
java -version
```

This notebook is read-only with respect to `captioning/` — it only loads
your trained checkpoint, nothing about `app.py` / `predict.py` / `train.py`
changes because of this file existing.


In [14]:

# --- Configuration -----------------------------------------------------
# Set LIMIT to a small number (e.g. 50) for a fast smoke test before
# committing to the full clean-test run.
LIMIT = None            # None = full clean (leak-free) Karpathy test subset
BEAM_SIZE = 5            # matches predict.py's default and common literature setting
INCLUDE_SPICE = False    # SPICE needs an extra ~2GB Stanford CoreNLP model download

# If your clean (held-out) test set ends up smaller than this, you'll get a
# loud warning — a very small N makes BLEU/CIDEr noisy and hard to compare
# against papers that always report on the full 5,000-image test split.
MIN_CLEAN_SAMPLE_WARN = 1000


In [15]:

# --- Imports -------------------------------------------------------------
from __future__ import annotations

import io
import json
import os
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from PIL import Image
from tqdm.auto import tqdm

# Make the project package importable regardless of where this notebook
# lives, as long as it's placed at the project root (next to app.py).
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from captioning.checkpoint import load_checkpoint
from captioning.config import get_default_config
from captioning.dataset import eval_transform
from captioning.inference import beam_search_caption
from captioning.model import build_models
from captioning.vocab import Vocabulary

sys.modules["__main__"].Vocabulary = Vocabulary  # same unpickling alias predict.py/app.py use

CFG = get_default_config()
print("Project root:", PROJECT_ROOT)
print("Device:", CFG.device)


Project root: d:\Projects\newone
Device: cuda


In [3]:

import torch, sys
print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

python: c:\Users\yokes\anaconda3\envs\image_caption\python.exe
torch: 2.5.1+cu121
cuda_available: True
device_count: 1
gpu: NVIDIA GeForce GTX 1650 Ti


In [4]:

# --- Check Java is available (required by PTBTokenizer + METEOR) --------
try:
    out = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print((out.stderr or out.stdout).strip().splitlines()[0])
except FileNotFoundError:
    raise RuntimeError(
        "Java was not found on PATH. pycocoevalcap's tokenizer and METEOR "
        "scorer need a Java runtime to run their bundled .jar files. "
        "Install a JRE (e.g. Eclipse Temurin / OpenJDK) and re-run this cell."
    )


java version "21.0.1" 2023-10-17 LTS


## 1. Download the Karpathy split

Andrej Karpathy's own `dataset_coco.json` — defines which 113,287 / 5,000 /
5,000 images belong to train/val/test, plus 5 human reference captions per
image, plus (critically for this notebook) each image's original COCO2014
`filepath` (`train2014`/`val2014`) and `filename` — cached locally after the
first run.


In [5]:

KARPATHY_DIR = PROJECT_ROOT / "data" / "coco" / "karpathy"
KARPATHY_JSON = KARPATHY_DIR / "dataset_coco.json"
KARPATHY_ZIP_URL = "https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip"

KARPATHY_DIR.mkdir(parents=True, exist_ok=True)

if not KARPATHY_JSON.is_file():
    print(f"Downloading Karpathy split from {KARPATHY_ZIP_URL} ...")
    req = urllib.request.Request(KARPATHY_ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        zip_bytes = resp.read()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        with zf.open("dataset_coco.json") as src, open(KARPATHY_JSON, "wb") as dst:
            dst.write(src.read())
    print("Saved to", KARPATHY_JSON)
else:
    print("Already downloaded:", KARPATHY_JSON)

with open(KARPATHY_JSON, "r", encoding="utf-8") as f:
    karpathy = json.load(f)

test_entries = [im for im in karpathy["images"] if im["split"] == "test"]
print(f"Karpathy test split: {len(test_entries)} images (expect 5,000)")


Already downloaded: d:\Projects\newone\data\coco\karpathy\dataset_coco.json
Karpathy test split: 5000 images (expect 5,000)


## 2. Detect contamination — the actual leakage check

This is the step the original notebook was missing. We load the **exact
annotation file the model was trained on** (`CFG.train_ann`, i.e.
`captions_train2017.json`) and check, for every Karpathy-test image, whether
its `cocoid` also appears there.

If it does, that image (and its caption) was almost certainly part of the
model's training data — scoring on it tells you nothing about
generalization, only about memorization. We also check `CFG.val_ann`
(`captions_val2017.json`) separately: unless your `train.py` also trains on
the validation split (uncommon, but worth double-checking in your own
training code), overlap with `val_ann` is *not* leakage.


In [6]:

def _load_id_to_file(ann_path: Path) -> dict[int, str]:
    with open(ann_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return {im["id"]: im["file_name"] for im in data["images"]}

train2017_ids = _load_id_to_file(Path(CFG.train_ann))
val2017_ids = _load_id_to_file(Path(CFG.val_ann))

# NOTE: if your train.py trains on train2017 + val2017 combined, add
# val2017_ids into the contamination check below too. As shipped, this
# assumes only train_ann was used for training (the common setup).
contaminated_ids = {e["cocoid"] for e in test_entries if e["cocoid"] in train2017_ids}
clean_entries = [e for e in test_entries if e["cocoid"] not in contaminated_ids]

n_total = len(test_entries)
n_contam = len(contaminated_ids)
n_clean = len(clean_entries)

print("=" * 50)
print("CONTAMINATION CHECK")
print("=" * 50)
print(f"Karpathy test images total       : {n_total}")
print(f"Found in your training annotation: {n_contam}  ({n_contam / n_total:.1%})  <- must be EXCLUDED")
print(f"Truly held-out (clean) images     : {n_clean}  ({n_clean / n_total:.1%})")
print("=" * 50)

if n_contam == 0:
    print("No overlap detected — your earlier run was NOT contaminated by this "
          "particular issue. The inflated score has some other cause; worth "
          "double-checking for duplicate/near-duplicate images between train "
          "and test, or a bug in how captions were paired with images.")
elif n_clean < MIN_CLEAN_SAMPLE_WARN:
    print(f"\nWARNING: only {n_clean} clean images remain — well below the "
          f"full 5,000 papers report on. Scores below are directionally "
          f"honest but noisier / less directly comparable than a full "
          f"5,000-image run. Consider retraining on a strict COCO2014 "
          f"Karpathy-train split (see notebook markdown at the top) to get "
          f"a full, clean 5,000-image test set.")


CONTAMINATION CHECK
Karpathy test images total       : 5000
Found in your training annotation: 4407  (88.1%)  <- must be EXCLUDED
Truly held-out (clean) images     : 593  (11.9%)



## 3. Resolve image paths for the CLEAN set only

For each held-out image we try, in order:

1. `val2017/` — if the id is in `captions_val2017.json` (safe: not used for
   training here).
2. Local `train2017/` folder — **only reachable for ids we already excluded
   above**, so this branch will never actually fire for `clean_entries`; it's
   kept only as a defensive assertion.
3. Official **COCO2014** CDN, via the `filepath` (`train2014`/`val2014`) and
   `filename` fields that ship inside Karpathy's `dataset_coco.json` itself —
   this covers images that COCO2017 dropped from its annotations entirely.


In [7]:

COCO_IMG_BASE = "http://images.cocodataset.org"

CLEAN_IMG_DIR = PROJECT_ROOT / "data" / "coco" / "karpathy_clean_test"
CLEAN_IMG_DIR.mkdir(parents=True, exist_ok=True)

val_dir = Path(CFG.val_images)
train_dir = Path(CFG.train_images)

resolved, missing = [], []

for entry in clean_entries:
    cocoid = entry["cocoid"]
    refs = [s["raw"].strip() for s in entry["sentences"]]

    # 1) safe local source: val2017 (not used for training)
    if cocoid in val2017_ids:
        local_path = val_dir / val2017_ids[cocoid]
        if local_path.is_file():
            resolved.append({"image_id": cocoid, "path": local_path, "refs": refs})
            continue

    # 2) defensive check — should never trigger for clean_entries
    if cocoid in train2017_ids:
        raise RuntimeError(
            f"cocoid {cocoid} unexpectedly found in train2017_ids after "
            "contamination filtering — this should not happen."
        )

    # 3) fall back to the official COCO2014 CDN using Karpathy's own
    #    filepath/filename fields (covers ids absent from both 2017 files)
    fname = entry["filename"]
    subdir = entry.get("filepath", "val2014")  # 'train2014' or 'val2014'
    local_path = CLEAN_IMG_DIR / fname
    if not local_path.is_file():
        url = f"{COCO_IMG_BASE}/{subdir}/{fname}"
        try:
            urllib.request.urlretrieve(url, local_path)
        except Exception as e:
            missing.append((cocoid, str(e)))
            continue

    resolved.append({"image_id": cocoid, "path": local_path, "refs": refs})

print(f"Resolved (clean, leak-free): {len(resolved)} / {len(clean_entries)}")
print(f"Missing / failed to fetch  : {len(missing)}")
if missing:
    print("First few failures:", missing[:5])

if LIMIT:
    resolved = resolved[:LIMIT]
    print(f"LIMIT set — evaluating on {len(resolved)} images only (smoke test).")


Resolved (clean, leak-free): 593 / 593
Missing / failed to fetch  : 0


## 4. Load the trained model

Same checkpoint auto-detection as `app.py` / `predict.py` — looks for
`checkpoints/best_model.pth`, falling back to `checkpoints/last_model.pth`.


In [8]:

def pick_checkpoint() -> Path:
    for name in ("best_model.pth", "last_model.pth"):
        p = Path(CFG.checkpoint_dir) / name
        if p.is_file():
            return p
    raise FileNotFoundError(
        f"No checkpoint found in {CFG.checkpoint_dir}/. Train the model first, "
        "or place best_model.pth / last_model.pth there."
    )

ckpt_path = pick_checkpoint()
device = torch.device(CFG.device)
state = torch.load(ckpt_path, map_location=device, weights_only=False)

vocab = state["vocab"]
saved_cfg = state.get("config") or {}
for k in ("attention_dim", "embed_dim", "decoder_dim", "encoder_dim", "dropout", "max_caption_len"):
    if k in saved_cfg:
        setattr(CFG, k, saved_cfg[k])

encoder, decoder = build_models(CFG, len(vocab), device, pretrained=False)
load_checkpoint(str(ckpt_path), encoder, decoder, map_location=device)
encoder.eval()
decoder.eval()

print(f"Loaded {ckpt_path.name}  ·  device={device.type}  ·  vocab={len(vocab):,} words")


Loaded best_model.pth  ·  device=cuda  ·  vocab=10,307 words


## 5. Generate captions with beam search — CLEAN set only

Every image here was verified in Step 2/3 to be absent from the model's
training annotation. This is the slow part; interrupt and re-run with a
smaller `LIMIT` above to gauge timing first if you like.


In [9]:

gts_raw: dict[str, list[dict]] = {}
res_raw: dict[str, list[dict]] = {}

t0 = time.time()
with torch.no_grad():
    for item in tqdm(resolved, desc=f"beam_k{BEAM_SIZE} (clean)"):
        img_id = str(item["image_id"])
        image = Image.open(item["path"]).convert("RGB")
        tensor = eval_transform(image)

        caption, _ids, _alphas = beam_search_caption(
            encoder, decoder, vocab, tensor, device, CFG.encoder_dim,
            beam_size=BEAM_SIZE, max_len=CFG.max_caption_len,
        )

        res_raw[img_id] = [{"caption": caption}]
        gts_raw[img_id] = [{"caption": r} for r in item["refs"]]

elapsed = time.time() - t0
print(f"Generated {len(res_raw)} captions in {elapsed/60:.1f} min "
      f"({elapsed/max(len(res_raw),1):.2f}s/image)")


beam_k5 (clean): 100%|██████████| 593/593 [01:15<00:00,  7.87it/s]

Generated 593 captions in 1.3 min (0.13s/image)


## 6. Score with the official toolkit

Same classes, same call pattern as `pycocoevalcap`'s own
`COCOEvalCap.evaluate()` — tokenize with Stanford `PTBTokenizer`, then
`Bleu`, `Meteor`, `Rouge`, `Cider` (and optionally `Spice`). Only the input
image set has changed from the original notebook — the scoring code itself
is untouched, so this stays directly comparable to how papers compute these
metrics.


In [10]:

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

print("Tokenizing with Stanford PTBTokenizer...")
tokenizer = PTBTokenizer()
gts = tokenizer.tokenize(gts_raw)
res = tokenizer.tokenize(res_raw)

results = {}

print("Scoring BLEU-1..4 ...")
bleu_scores, _ = Bleu(4).compute_score(gts, res, verbose=0)
for i, s in enumerate(bleu_scores, start=1):
    results[f"BLEU-{i}"] = s

print("Scoring METEOR ...")
results["METEOR"], _ = Meteor().compute_score(gts, res)

print("Scoring ROUGE-L ...")
results["ROUGE-L"], _ = Rouge().compute_score(gts, res)

print("Scoring CIDEr ...")
results["CIDEr"], _ = Cider().compute_score(gts, res)

if INCLUDE_SPICE:
    from pycocoevalcap.spice.spice import Spice
    print("Scoring SPICE (slower, downloads Stanford CoreNLP models on first use) ...")
    results["SPICE"], _ = Spice().compute_score(gts, res)

print()
print("=" * 40)
print(f"CLEAN / LEAK-FREE SCORES  (N = {len(res_raw)} held-out images)")
print("=" * 40)
for k, v in results.items():
    print(f"{k:10s}: {v:.4f}")
print("=" * 40)


Tokenizing with Stanford PTBTokenizer...
Scoring BLEU-1..4 ...
Scoring METEOR ...
Scoring ROUGE-L ...
Scoring CIDEr ...

CLEAN / LEAK-FREE SCORES  (N = 593 held-out images)
BLEU-1    : 0.7488
BLEU-2    : 0.5893
BLEU-3    : 0.4564
BLEU-4    : 0.3541
METEOR    : 0.2647
ROUGE-L   : 0.5613
CIDEr     : 1.1288


## 7. Compare: contaminated run vs. clean run vs. published papers

The middle row (`This model — ORIGINAL RUN, contaminated`) is the score
from `evaluate_karpathy.ipynb`, kept here only so the gap is visible — it
should **not** be reported anywhere as your model's real performance.

Reference rows are the commonly-cited Karpathy-test-split numbers as
reported in the papers themselves (or in later papers' comparison tables,
the standard citation format once evaluation protocols converged).


In [11]:

import pandas as pd

published = {
    "Show and Tell, NIC (2015)":                         {"BLEU-4": 0.183, "METEOR": 0.237, "CIDEr": 0.855},
    "Show, Attend and Tell — hard attn (2015)":          {"BLEU-4": 0.250, "METEOR": 0.230, "CIDEr": None},
    "SCST / Att2all, RL-optimized (2017)":               {"BLEU-4": 0.342, "METEOR": 0.257, "CIDEr": 1.140},
    "Up-Down / BUTD, RL-optimized (2018)":                {"BLEU-4": 0.369, "METEOR": 0.276, "CIDEr": 1.179},
    "AoANet (2019)":                                      {"BLEU-4": 0.379, "METEOR": 0.291, "CIDEr": 1.298},
    "M2-Transformer, single model (2020)":                {"BLEU-4": 0.393, "METEOR": 0.293, "CIDEr": 1.312},
    "VinVL (2021)":                                       {"BLEU-4": 0.410, "METEOR": 0.310, "CIDEr": 1.308},
    "BLIP (2022)":                                        {"BLEU-4": None,  "METEOR": 0.297, "CIDEr": 1.336},
    "GIT (2022, large-scale pretraining)":                {"BLEU-4": 0.441, "METEOR": 0.315, "CIDEr": 1.448},
    "BLIP-2 (2023, large-scale pretraining)":             {"BLEU-4": 0.428, "METEOR": None,  "CIDEr": 1.456},
}

# scores you originally posted, from the CONTAMINATED run — kept for
# transparency / comparison only, not a valid result
original_contaminated = {"BLEU-4": 0.4165, "METEOR": 0.2921, "CIDEr": 1.2757}

rows = []
for name, m in published.items():
    rows.append({"model": name, "BLEU-4": m["BLEU-4"], "METEOR": m["METEOR"], "CIDEr": m["CIDEr"]})

rows.append({
    "model": "This model — ORIGINAL RUN, CONTAMINATED (do not cite)",
    "BLEU-4": original_contaminated["BLEU-4"],
    "METEOR": original_contaminated["METEOR"],
    "CIDEr": original_contaminated["CIDEr"],
})

rows.append({
    "model": f"This model — CLEAN RUN (checkpoint: {ckpt_path.name}, beam={BEAM_SIZE}, N={len(res_raw)})",
    "BLEU-4": round(results["BLEU-4"], 3),
    "METEOR": round(results["METEOR"], 3),
    "CIDEr": round(results["CIDEr"], 3),
})

df = pd.DataFrame(rows).set_index("model")
df


,BLEU-4,METEOR,CIDEr
model,,,
"Show and Tell, NIC (2015)",0.1830,0.2370,0.8550
"Show, Attend and Tell — hard attn (2015)",0.2500,0.2300,NaN
"SCST / Att2all, RL-optimized (2017)",0.3420,0.2570,1.1400
"Up-Down / BUTD, RL-optimized (2018)",0.3690,0.2760,1.1790
AoANet (2019),0.3790,0.2910,1.2980
"M2-Transformer, single model (2020)",0.3930,0.2930,1.3120
VinVL (2021),0.4100,0.3100,1.3080
BLIP (2022),NaN,0.2970,1.3360
"GIT (2022, large-scale pretraining)",0.4410,0.3150,1.4480


## 8. Save results

Written to `outputs/eval_karpathy_clean/` — kept separate from the
original (contaminated) notebook's output folder, and includes the
contamination diagnostics so the provenance of the number is never lost.


In [12]:

out_dir = PROJECT_ROOT / "outputs" / "eval_karpathy_clean"
out_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "checkpoint": ckpt_path.name,
    "beam_size": BEAM_SIZE,
    "karpathy_test_total": n_total,
    "contaminated_excluded": n_contam,
    "contamination_pct": round(n_contam / n_total, 4),
    "num_images_evaluated_clean": len(res_raw),
    "scores_clean": results,
    "scores_original_contaminated_DO_NOT_CITE": original_contaminated,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

out_path = out_dir / f"karpathy_eval_clean_{time.strftime('%Y%m%d_%H%M%S')}.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", out_path)


Saved: d:\Projects\newone\outputs\eval_karpathy_clean\karpathy_eval_clean_20260809_222556.json


In [13]:
import json

with open("data/coco/karpathy/dataset_coco.json") as f:
    karpathy = json.load(f)
test_ids = {im["cocoid"] for im in karpathy["images"] if im["split"] == "test"}

with open("data/coco/annotations/captions_train2017.json") as f:
    train_ann = json.load(f)
train_ids = {im["id"] for im in train_ann["images"]}

overlap = test_ids & train_ids
print(f"Karpathy test images: {len(test_ids)}")
print(f"Overlapping with your train2017 training set: {len(overlap)} ({len(overlap)/len(test_ids):.1%})")

Karpathy test images: 5000
Overlapping with your train2017 training set: 4407 (88.1%)


In [18]:
# ============================================================
# LEAK-FREE EVAL: score only on Karpathy images that live in
# val2017 — the one folder your training pipeline never reads.
# ============================================================
import json, time
from pathlib import Path
from PIL import Image
import torch
from tqdm.auto import tqdm

# --- 1. Load the Karpathy split -----------------------------------------
KARPATHY_JSON = Path(CFG.data_root) / "karpathy" / "dataset_coco.json"
with open(KARPATHY_JSON, "r", encoding="utf-8") as f:
    karpathy = json.load(f)

# Karpathy's own val + test buckets are both eligible — neither was ever
# read by your train.ipynb (which only touches train_ann/train_images).
karpathy_entries = [im for im in karpathy["images"] if im["split"] in ("val", "test")]
print(f"Karpathy val+test images: {len(karpathy_entries)}")

# --- 2. Load id -> filename maps for both 2017 annotation files ---------
def _load_id_to_file(ann_path):
    with open(ann_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return {im["id"]: im["file_name"] for im in data["images"]}

train2017_ids = _load_id_to_file(CFG.train_ann)   # used for training — exclude any overlap
val2017_ids   = _load_id_to_file(CFG.val_ann)     # NEVER used for training — safe

# --- 3. Keep only images that are (a) physically in val2017 AND
#        (b) NOT also present in the training annotation ----------------
clean_entries = []
skipped_in_train = 0
for entry in karpathy_entries:
    cocoid = entry["cocoid"]
    if cocoid in train2017_ids:
        skipped_in_train += 1          # would be leakage — drop it
        continue
    if cocoid not in val2017_ids:
        continue                        # not locally available in val2017 either
    fname = val2017_ids[cocoid]
    local_path = Path(CFG.val_images) / fname
    if not local_path.is_file():
        continue
    refs = [s["raw"].strip() for s in entry["sentences"]]
    clean_entries.append({"image_id": cocoid, "path": local_path, "refs": refs})

print(f"Also present in train2017 (excluded as leakage): {skipped_in_train}")
print(f"Clean, held-out, locally available in val2017   : {len(clean_entries)}")

if len(clean_entries) < 500:
    print("WARNING: small clean sample — scores will be noisier than a full 5,000-image run.")

# --- 4. Load checkpoint (same as your existing eval notebook) -----------
def pick_checkpoint():
    for name in ("best_model.pth", "last_model.pth"):
        p = Path(CFG.checkpoint_dir) / name
        if p.is_file():
            return p
    raise FileNotFoundError(f"No checkpoint found in {CFG.checkpoint_dir}/")

ckpt_path = pick_checkpoint()
device = torch.device(CFG.device)
state = torch.load(ckpt_path, map_location=device, weights_only=False)
vocab = state["vocab"]
saved_CFG = state.get("config") or {}
for k in ("attention_dim", "embed_dim", "decoder_dim", "encoder_dim", "dropout", "max_caption_len"):
    if k in saved_CFG:
        setattr(CFG, k, saved_CFG[k])

encoder, decoder = build_models(CFG, len(vocab), device, pretrained=False)
load_checkpoint(str(ckpt_path), encoder, decoder, map_location=device)
encoder.eval(); decoder.eval()
print(f"Loaded {ckpt_path.name}  ·  device={device.type}  ·  vocab={len(vocab):,} words")

# --- 5. Generate captions with beam search on the clean set --------------
BEAM_SIZE = 5
gts_raw, res_raw = {}, {}

t0 = time.time()
with torch.no_grad():
    for item in tqdm(clean_entries, desc=f"beam_k{BEAM_SIZE} (val2017, clean)"):
        img_id = str(item["image_id"])
        image = Image.open(item["path"]).convert("RGB")
        tensor = eval_transform(image)
        caption, _ids, _alphas = beam_search_caption(
            encoder, decoder, vocab, tensor, device, CFG.encoder_dim,
            beam_size=BEAM_SIZE, max_len=CFG.max_caption_len,
        )
        res_raw[img_id] = [{"caption": caption}]
        gts_raw[img_id] = [{"caption": r} for r in item["refs"]]

elapsed = time.time() - t0
print(f"Generated {len(res_raw)} captions in {elapsed/60:.1f} min "
      f"({elapsed/max(len(res_raw),1):.2f}s/image)")

# --- 6. Score with the official toolkit -----------------------------------
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

tokenizer = PTBTokenizer()
gts = tokenizer.tokenize(gts_raw)
res = tokenizer.tokenize(res_raw)

results = {}
bleu_scores, _ = Bleu(4).compute_score(gts, res, verbose=0)
for i, s in enumerate(bleu_scores, start=1):
    results[f"BLEU-{i}"] = s
results["METEOR"], _ = Meteor().compute_score(gts, res)
results["ROUGE-L"], _ = Rouge().compute_score(gts, res)
results["CIDEr"], _ = Cider().compute_score(gts, res)

print()
print("=" * 45)
print(f"CLEAN SCORES — val2017-sourced, N={len(res_raw)} (0% train overlap)")
print("=" * 45)
for k, v in results.items():
    print(f"{k:10s}: {v:.4f}")
print("=" * 45)

Karpathy val+test images: 10000
Also present in train2017 (excluded as leakage): 8799
Clean, held-out, locally available in val2017   : 1201
Loaded best_model.pth  ·  device=cuda  ·  vocab=10,307 words


beam_k5 (val2017, clean): 100%|██████████| 1201/1201 [02:21<00:00,  8.51it/s]


Generated 1201 captions in 2.4 min (0.12s/image)

CLEAN SCORES — val2017-sourced, N=1201 (0% train overlap)
BLEU-1    : 0.7471
BLEU-2    : 0.5895
BLEU-3    : 0.4571
BLEU-4    : 0.3535
METEOR    : 0.2648
ROUGE-L   : 0.5580
CIDEr     : 1.1033
